An example that LLM sends back tool_use with multiple tool calls

disable_parallel_tool_use does the trick

```
response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    tools=tools,
    # Ask for at most one tool call per turn.
    tool_choice={"type": "auto", "disable_parallel_tool_use": False},
    messages=messages,
)
```

In [2]:
from util.claude_client import get_claude_client_and_default_model

client = get_claude_client_and_default_model('deepseek')





llm profile name : deepseek , model : deepseek-flash


In [3]:
tools = [
    {
        'name': 'get_time',
        'description' : 'find the current time in a given timezone',
        'input_schema': {
            'type': 'object',
            'properties': {
                'timezone': {
                    'type': 'string',
                    'description' : 'time zone abbreviations, e.g. EST' ,
                    'default': 'UTC'},
            },
            'required': ['timezone']

        }

    },

    {
        'name': 'get_weather',
        'description': 'Get the current weather in a given location',
        'input_schema': {
            'type': 'object',
            'properties': {
                'location': {
                    'type': 'string',
                    'description' : 'The city and state, e.g. San Francisco, CA',
                }
            }
        },
        'required': ['location']
    }

]
messages = [
    {
        "role": "user",
        "content": "What's the weather in SF and NYC, and what time is it there?",
    }
]
print(messages)

def call_tool( tool_name : str , input:dict ):
    print( f'call tool : tool name: {tool_name}  , input: {input}')
    if tool_name == 'get_time':
        if "PDT" in str(input['timezone']):
            result = "2:30 PM PDT"
        elif "PST" in str(input['timezone']):
            result = "1:30 PM PDT"
        elif "EDT" in str(input['timezone']):
            result = "5:30 PM EDT"
        else:
            result = "4:30 PM EST"
    else:
        if "San Francisco" in str(input['location']):
            result = "San Francisco: 68°F, partly cloudy"
        else:
            result = "New York: 45°F, clear skies"
    return result


[{'role': 'user', 'content': "What's the weather in SF and NYC, and what time is it there?"}]


LLM sends back 4 tool executions in one message , all too execution results need to be uploaded with same message instead of 4 user messages

In [4]:
client  , llm= get_claude_client_and_default_model('deepseek')
while True:

    print("messages ######################### \n")
    print(messages)
    message = client.messages.create(
        max_tokens=1024,
        model=llm,
        messages = messages,
        tools=tools,
        tool_choice={ "type":"auto" , "disable_parallel_tool_use": False},
    )
    print( message.model_dump_json(indent=4) )
    if message.stop_reason == 'end_turn':
        break


    messages += [

        {
            "role": "assistant",
            "content": message.content
        }

    ]




    tool_uses = [block for block in message.content if block.type == 'tool_use']
    print(f"number of tools used: {len(tool_uses)}")
    tool_execution_results = []
    for tool_use in tool_uses:
        tool_execution_result = call_tool(tool_use.name, tool_use.input)

        tool_execution_results.append(
            {
                    'type':'tool_result',
                    'tool_use_id': tool_use.id,
                    'content': tool_execution_result
            }
        )

    messages += [

        {
            'role': 'user',
            'content':tool_execution_results
        }

    ]

print( message.model_dump_json(indent=4) )


llm profile name : deepseek , model : deepseek-flash
messages ######################### 

[{'role': 'user', 'content': "What's the weather in SF and NYC, and what time is it there?"}]
{
    "id": "43b9e2a7-2f11-457b-8332-9ec1905db5ed",
    "container": null,
    "content": [
        {
            "signature": "43b9e2a7-2f11-457b-8332-9ec1905db5ed",
            "thinking": "I need to get weather and time for both SF and NYC. These are independent calls, so I'll make them all in one block.\n\nSan Francisco is Pacific Time (PT), NYC is Eastern Time (ET).",
            "type": "thinking"
        },
        {
            "citations": null,
            "text": "I'll get the weather and time for both cities.",
            "type": "text"
        },
        {
            "id": "call_00_PlMNd2733LXJ2ychKKrH5734",
            "caller": null,
            "input": {
                "location": "San Francisco, CA"
            },
            "name": "get_weather",
            "type": "tool_use",
    